# API RAG QA Pipeline

This notebook builds a RAG pipeline for the PDF-derived Markdown corpus and answers the questions in `qa_set.csv`.

It is designed for a non-local/cloud notebook environment:

- Put the converted Markdown files and `qa_set.csv` in the same folder as this notebook.
- Set `CSCS_API_KEY` in the environment, or create a file named `environment` containing `CSCS_API_KEY=...`.
- The LLM answer generation uses the Swiss AI OpenAI-compatible API.

Retrieval settings used below:

- Chunk size: `250` words
- Chunk overlap: `50` words
- Cosine similarity threshold: `0.15`
- Maximum context chunks sent to the model: `6`
- Minimum fallback chunks: `3`


## 1. Install and Import Dependencies

In [1]:
# Run this cell once in a fresh notebook environment.
%pip install -q openai python-dotenv pandas scikit-learn numpy tqdm


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import csv
import json
import os
import re
import sqlite3
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

load_dotenv(dotenv_path="environment")

WORK_DIR = Path.cwd()
QA_CSV = WORK_DIR / "qa_set.csv"
OUTPUT_CSV = WORK_DIR / "qa_set_api_rag_answers.csv"
DB_PATH = WORK_DIR / "rag_tfidf_database.sqlite"

CHUNK_WORDS = 250
OVERLAP_WORDS = 50
SIMILARITY_THRESHOLD = 0.15
MAX_CONTEXT_CHUNKS = 6
MIN_CONTEXT_CHUNKS = 3

API_BASE_URL = "https://api.swissai.svc.cscs.ch/v1"
MODEL_NAME = "moonshotai/Kimi-K2.5-SDSC"  # Strongest model listed in the API example.
# Fast alternative: "zai-org/GLM-4.7-Flash"
# Open model alternatives: "swiss-ai/Apertus-8B-Instruct-2509", "swiss-ai/Apertus-70B-Instruct-2509"

client = OpenAI(
    base_url=API_BASE_URL,
    api_key=os.getenv("CSCS_API_KEY"),
)

print("Working directory:", WORK_DIR)
print("qa_set.csv exists:", QA_CSV.exists())
print("API key configured:", bool(os.getenv("CSCS_API_KEY")))

Working directory: /home/renku/work/Durham-Hackathon-2026-w2t1
qa_set.csv exists: True
API key configured: True


## 2. Load Markdown Corpus

In [6]:
IMAGE_PATTERN = re.compile(r"!\[[^\]]*\]\([^)]+\)")
WORD_PATTERN = re.compile(r"\S+")


def clean_markdown(text: str) -> str:
    text = IMAGE_PATTERN.sub(" ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"[ \t]+", " ", text)
    return text


markdown_files = sorted(
    path for path in WORK_DIR.glob("*.md")
    if path.name.lower() not in {"readme.md"}
)

if not markdown_files:
    raise FileNotFoundError(
        "No .md files found. Put the PDF-derived Markdown files in the same folder as this notebook."
    )

print(f"Found {len(markdown_files)} Markdown files:")
for path in markdown_files:
    print("-", path.name)

Found 4 Markdown files:
- Web Version _E-Government Survey 2024 11102024.md
- World_Inequality_Report_2026.md
- natural-catastrophe-and-climate-report-2023.md
- swissre_sigma-1_2024_english.md


## 3. Chunk the Corpus

In [7]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: int
    source: str
    chunk_index: int
    start_word: int
    end_word: int
    text: str


def chunk_text(source: str, text: str, chunk_words: int, overlap_words: int, start_id: int) -> list[Chunk]:
    if overlap_words >= chunk_words:
        raise ValueError("Overlap must be smaller than chunk size.")

    words = WORD_PATTERN.findall(clean_markdown(text))
    if not words:
        return []

    chunks = []
    step = chunk_words - overlap_words
    for chunk_index, start in enumerate(range(0, len(words), step)):
        end = min(start + chunk_words, len(words))
        chunks.append(
            Chunk(
                chunk_id=start_id + len(chunks),
                source=source,
                chunk_index=chunk_index,
                start_word=start,
                end_word=end,
                text=" ".join(words[start:end]),
            )
        )
        if end == len(words):
            break
    return chunks


chunks: list[Chunk] = []
next_id = 1
for md_path in markdown_files:
    text = md_path.read_text(encoding="utf-8", errors="replace")
    file_chunks = chunk_text(md_path.name, text, CHUNK_WORDS, OVERLAP_WORDS, next_id)
    chunks.extend(file_chunks)
    next_id += len(file_chunks)

print(f"Created {len(chunks)} chunks.")
pd.DataFrame([c.__dict__ for c in chunks[:5]])

Created 1044 chunks.


,chunk_id,source,chunk_index,start_word,end_word,text
0,1,Web Version _E-Government Survey 2024 11102024.md,0,0,250,## **E-Government Survey 2024** Accelerating D...
1,2,Web Version _E-Government Survey 2024 11102024.md,1,200,450,‘country’ and ‘economy’ as used in this Report...
2,3,Web Version _E-Government Survey 2024 11102024.md,2,400,650,"or its senior management, or of the experts wh..."
3,4,Web Version _E-Government Survey 2024 11102024.md,3,600,850,2024 UN E-GovErNmENt SUrvEy iv PREFAcE ## **Pr...
4,5,Web Version _E-Government Survey 2024 11102024.md,4,800,1050,is crucial for comprehensive digital transform...


## 4. Build Embeddings and Save the RAG Database

In [8]:
# TF-IDF embeddings are deterministic, fast, and work in a cloud notebook without local LLM inference.
# Cosine similarity is computed over these embeddings.

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=80_000,
    min_df=1,
)

chunk_texts = [chunk.text for chunk in chunks]
chunk_matrix = vectorizer.fit_transform(chunk_texts)

print("Embedding matrix shape:", chunk_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Embedding matrix shape: (1044, 80000)
Vocabulary size: 80000


In [9]:
def save_database(db_path: Path) -> None:
    if db_path.exists():
        db_path.unlink()

    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(
            """
            CREATE TABLE metadata (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL
            );

            CREATE TABLE chunks (
                id INTEGER PRIMARY KEY,
                source TEXT NOT NULL,
                chunk_index INTEGER NOT NULL,
                start_word INTEGER NOT NULL,
                end_word INTEGER NOT NULL,
                text TEXT NOT NULL
            );

            CREATE INDEX idx_chunks_source ON chunks(source);
            """
        )
        metadata = {
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "chunk_words": CHUNK_WORDS,
            "overlap_words": OVERLAP_WORDS,
            "similarity_threshold": SIMILARITY_THRESHOLD,
            "max_context_chunks": MAX_CONTEXT_CHUNKS,
            "min_context_chunks": MIN_CONTEXT_CHUNKS,
            "embedding_backend": "sklearn-tfidf-word-1-2gram",
            "chunk_count": len(chunks),
            "sources": sorted({chunk.source for chunk in chunks}),
        }
        conn.execute(
            "INSERT INTO metadata(key, value) VALUES (?, ?)",
            ("index", json.dumps(metadata, indent=2)),
        )
        conn.executemany(
            """
            INSERT INTO chunks(id, source, chunk_index, start_word, end_word, text)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            [
                (
                    chunk.chunk_id,
                    chunk.source,
                    chunk.chunk_index,
                    chunk.start_word,
                    chunk.end_word,
                    chunk.text,
                )
                for chunk in chunks
            ],
        )
        conn.commit()
    finally:
        conn.close()


save_database(DB_PATH)
print("Saved database:", DB_PATH)

Saved database: /home/renku/work/Durham-Hackathon-2026-w2t1/rag_tfidf_database.sqlite


## 5. Retrieval With Cosine Similarity Threshold

In [10]:
def retrieve_chunks(
    question: str,
    threshold: float = SIMILARITY_THRESHOLD,
    max_chunks: int = MAX_CONTEXT_CHUNKS,
    min_chunks: int = MIN_CONTEXT_CHUNKS,
) -> list[dict]:
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, chunk_matrix).ravel()
    ranked_indices = np.argsort(scores)[::-1]

    selected = [
        {
            "chunk_id": chunks[i].chunk_id,
            "source": chunks[i].source,
            "chunk_index": chunks[i].chunk_index,
            "score": float(scores[i]),
            "text": chunks[i].text,
        }
        for i in ranked_indices
        if scores[i] >= threshold
    ][:max_chunks]

    # Fallback: if the threshold is too strict for a short/narrow question,
    # still provide the best few chunks so the model can answer or say not found.
    if len(selected) < min_chunks:
        fallback = [
            {
                "chunk_id": chunks[i].chunk_id,
                "source": chunks[i].source,
                "chunk_index": chunks[i].chunk_index,
                "score": float(scores[i]),
                "text": chunks[i].text,
            }
            for i in ranked_indices[:min_chunks]
        ]
        seen = {item["chunk_id"] for item in selected}
        selected.extend(item for item in fallback if item["chunk_id"] not in seen)

    return selected[:max_chunks]


sample_question = "How much higher are the 2023 insured losses than the previous 10 year average?"
sample_chunks = retrieve_chunks(sample_question)
[(c["score"], c["source"], c["chunk_index"]) for c in sample_chunks]

[(0.30275251596238184, 'swissre_sigma-1_2024_english.md', 16),
 (0.2613026064759836, 'swissre_sigma-1_2024_english.md', 12),
 (0.21706114729252715, 'swissre_sigma-1_2024_english.md', 21),
 (0.1919479831370779, 'swissre_sigma-1_2024_english.md', 4),
 (0.18614294405272064, 'swissre_sigma-1_2024_english.md', 14),
 (0.17711066061379332, 'swissre_sigma-1_2024_english.md', 1)]

## 6. API Answer Generation

In [13]:
SYSTEM_PROMPT = """You are a careful RAG question-answering assistant.
Use only the supplied context.
Answer concisely, preferably in one or two sentences.
If the context does not contain the answer, write exactly: Not found in the retrieved context.
When relevant, include exact years, percentages, counts, names, or figure numbers.
Do not invent citations. Do not mention chunks unless asked.
"""


def format_context(retrieved: list[dict], max_chars_per_chunk: int = 1600) -> str:
    parts = []
    for i, chunk in enumerate(retrieved, start=1):
        text = chunk["text"][:max_chars_per_chunk]
        parts.append(
            f"[Context {i}] source={chunk['source']} chunk={chunk['chunk_index']} cosine={chunk['score']:.3f}\n{text}"
        )
    return "\n\n".join(parts)


def answer_question(question: str, model: str = MODEL_NAME) -> tuple[str, list[dict]]:
    retrieved = retrieve_chunks(question)
    context = format_context(retrieved)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:",
            },
        ],
        temperature=0.0,
        max_tokens=180,
    )
    return extract_chat_message_text(response.choices[0].message), retrieved

In [15]:
def extract_chat_message_text(message) -> str:
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        return content.strip()

    data = message.model_dump() if hasattr(message, "model_dump") else {}
    for key in ("reasoning_content", "reasoning", "output_text", "refusal"):
        value = data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    return "Not found in the retrieved context."

In [16]:
# Single-question test.
# This cell requires CSCS_API_KEY to be configured.

if not os.getenv("CSCS_API_KEY"):
    print("Set CSCS_API_KEY before running API calls.")
else:
    answer, evidence = answer_question(sample_question)
    print("Question:", sample_question)
    print("Answer:", answer)
    print("Evidence:")
    for item in evidence:
        print(f"- {item['source']}#{item['chunk_index']} cosine={item['score']:.3f}")

Question: How much higher are the 2023 insured losses than the previous 10 year average?
Answer: Not found in the retrieved context.
Evidence:
- swissre_sigma-1_2024_english.md#16 cosine=0.303
- swissre_sigma-1_2024_english.md#12 cosine=0.261
- swissre_sigma-1_2024_english.md#21 cosine=0.217
- swissre_sigma-1_2024_english.md#4 cosine=0.192
- swissre_sigma-1_2024_english.md#14 cosine=0.186
- swissre_sigma-1_2024_english.md#1 cosine=0.177


## 7. Load QA CSV

In [17]:
def read_qa_set(path: Path) -> pd.DataFrame:
    # qa_set.csv uses semicolons: question;answer
    df = pd.read_csv(path, sep=";", keep_default_na=False)
    if "question" not in df.columns:
        raise ValueError("qa_set.csv must contain a 'question' column.")
    if "answer" not in df.columns:
        df["answer"] = ""
    df["question"] = df["question"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    return df[df["question"] != ""].reset_index(drop=True)


qa_df = read_qa_set(QA_CSV)
print("Questions:", len(qa_df))
print("Existing answers:", int((qa_df["answer"] != "").sum()))
qa_df.head()

Questions: 40
Existing answers: 5


,question,answer
0,What year where there most casualties from man...,"2002. In that year more than 10,000 casualties..."
1,In what year did the quantity of man-made disa...,Man-made disasters peaked in 2005
2,Between 1970 and 2023 what year in the data sh...,2023. There were 218 instances of natural cat...
3,What year between 1994 and 2023 had the most h...,2011 with 6.
4,How much higher are the 2023 insured losses th...,They are higher by 21%.


## 8. Answer All Questions

In [18]:
def answer_qa_dataframe(
    df: pd.DataFrame,
    preserve_existing: bool = True,
    sleep_seconds: float = 0.2,
) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        question = row["question"]
        existing_answer = row["answer"]

        retrieved = retrieve_chunks(question)
        if preserve_existing and existing_answer:
            answer = existing_answer
        else:
            answer, retrieved = answer_question(question)
            time.sleep(sleep_seconds)

        rows.append(
            {
                "question": question,
                "answer": answer,
                "retrieved_sources": " | ".join(
                    f"{item['source']}#{item['chunk_index']} ({item['score']:.3f})"
                    for item in retrieved
                ),
                "retrieved_chunk_ids": ",".join(str(item["chunk_id"]) for item in retrieved),
            }
        )
    return pd.DataFrame(rows)


if not os.getenv("CSCS_API_KEY"):
    print("Set CSCS_API_KEY before answering the full QA set.")
else:
    results_df = answer_qa_dataframe(qa_df, preserve_existing=True)
    results_df.to_csv(OUTPUT_CSV, sep=";", index=False)
    print("Wrote:", OUTPUT_CSV)
    display(results_df.head())

100%|██████████| 40/40 [03:44<00:00,  5.61s/it]

Wrote: /home/renku/work/Durham-Hackathon-2026-w2t1/qa_set_api_rag_answers.csv


,question,answer,retrieved_sources,retrieved_chunk_ids
0,What year where there most casualties from man...,"2002. In that year more than 10,000 casualties...",swissre_sigma-1_2024_english.md#79 (0.201) | s...,"1039,1032,1041"
1,In what year did the quantity of man-made disa...,Man-made disasters peaked in 2005,swissre_sigma-1_2024_english.md#72 (0.227) | s...,"1032,1033,1039"
2,Between 1970 and 2023 what year in the data sh...,2023. There were 218 instances of natural cat...,swissre_sigma-1_2024_english.md#72 (0.218) | s...,"1032,1033,981"
3,What year between 1994 and 2023 had the most h...,2011 with 6.,swissre_sigma-1_2024_english.md#23 (0.316) | s...,"983,982,981"
4,How much higher are the 2023 insured losses th...,They are higher by 21%.,swissre_sigma-1_2024_english.md#16 (0.303) | s...,"976,972,981,964,974,961"


In [22]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

display(results_df)

,question,answer,retrieved_sources,retrieved_chunk_ids
0,What year where there most casualties from man-made disasters in the recorded data?,"2002. In that year more than 10,000 casualties are ascribed to man-made catastrophes.",swissre_sigma-1_2024_english.md#79 (0.201) | swissre_sigma-1_2024_english.md#72 (0.162) | swissre_sigma-1_2024_english.md#81 (0.123),"1039,1032,1041"
1,In what year did the quantity of man-made disasters peak in the recorded data between 1970 and 2023?,Man-made disasters peaked in 2005,swissre_sigma-1_2024_english.md#72 (0.227) | swissre_sigma-1_2024_english.md#73 (0.178) | swissre_sigma-1_2024_english.md#79 (0.150),"1032,1033,1039"
2,Between 1970 and 2023 what year in the data shows the largest number of natural catastrophes?,2023. There were 218 instances of natural catastrophes.,swissre_sigma-1_2024_english.md#72 (0.218) | swissre_sigma-1_2024_english.md#73 (0.146) | swissre_sigma-1_2024_english.md#21 (0.144),"1032,1033,981"
3,What year between 1994 and 2023 had the most high severity ($5 billion in damages or more) natural catastrophes?,2011 with 6.,swissre_sigma-1_2024_english.md#23 (0.316) | swissre_sigma-1_2024_english.md#22 (0.270) | swissre_sigma-1_2024_english.md#21 (0.255),"983,982,981"
4,How much higher are the 2023 insured losses than the previous 10 year average?,They are higher by 21%.,swissre_sigma-1_2024_english.md#16 (0.303) | swissre_sigma-1_2024_english.md#12 (0.261) | swissre_sigma-1_2024_english.md#21 (0.217) | swissre_sigma-1_2024_english.md#4 (0.192) | swissre_sigma-1_2024_english.md#14 (0.186) | swissre_sigma-1_2024_english.md#1 (0.177),"976,972,981,964,974,961"
5,"Which figure shows the trend in insured losses over data from 1994 to 2023? In this figure, what is the highest insured loss year on record?",Not found in the retrieved context.,swissre_sigma-1_2024_english.md#20 (0.197) | swissre_sigma-1_2024_english.md#33 (0.194) | natural-catastrophe-and-climate-report-2023.md#25 (0.188) | swissre_sigma-1_2024_english.md#27 (0.183) | swissre_sigma-1_2024_english.md#7 (0.179) | swissre_sigma-1_2024_english.md#19 (0.178),"980,993,843,987,967,979"
6,"Before 2023, what was the highest year on record for European Severe Convective Storm losses?",Not found in the retrieved context.,natural-catastrophe-and-climate-report-2023.md#18 (0.185) | swissre_sigma-1_2024_english.md#77 (0.182) | natural-catastrophe-and-climate-report-2023.md#79 (0.181) | natural-catastrophe-and-climate-report-2023.md#22 (0.158) | swissre_sigma-1_2024_english.md#2 (0.156) | swissre_sigma-1_2024_english.md#30 (0.156),"836,1037,897,840,962,990"
7,What regions does the Swiss Re report on natural catastrophes split the US into for severe convective storm risk?,Not found in the retrieved context.,natural-catastrophe-and-climate-report-2023.md#18 (0.182) | natural-catastrophe-and-climate-report-2023.md#22 (0.170) | swissre_sigma-1_2024_english.md#77 (0.153),"836,840,1037"
8,What is the highest Benefit to Cost ratio building code element described in the Swiss Re report on natural catastrophes?,Not found in the retrieved context.,swissre_sigma-1_2024_english.md#64 (0.258) | swissre_sigma-1_2024_english.md#65 (0.122) | swissre_sigma-1_2024_english.md#79 (0.095),"1024,1025,1039"
9,"According to the Swiss Re Institute report on natural catastrophes, 2017 was a standout year in terms of insured loss damages. What were the names of the weather events which contributed most to this figure?",Not found in the retrieved context.,swissre_sigma-1_2024_english.md#20 (0.211) | swissre_sigma-1_2024_english.md#26 (0.200) | swissre_sigma-1_2024_english.md#19 (0.184) | natural-catastrophe-and-climate-report-2023.md#22 (0.166) | swissre_sigma-1_2024_english.md#11 (0.165) | swissre_sigma-1_2024_english.md#24 (0.161),"980,986,979,840,971,984"


In [32]:
TEAM_NAME = "Matterhorn"

In [33]:
import requests

BASE_URL = "http://durham-leaderboard-runai-innovation-klemen.inference.compute.datascience.ch"
leaderboard_endpoint = f"{BASE_URL}/api/v1/leaderboard"
submit_endpoint = f"{BASE_URL}/api/v1/submit"


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(leaderboard_endpoint).json()
    return pd.DataFrame(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str) -> pd.Series:
    if user == "your_team_name" or user == "":
        raise ValueError("Please set your team name in the 'TEAM_NAME' variable.")
    predictions = df[["question", "answer"]].to_dict(orient="records")
    submission = {"predictions": predictions}
    response = requests.post(submit_endpoint, json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        response.raise_for_status()
    print(response.json())
    return pd.Series(response.json())

In [34]:
submit(results_df, user=TEAM_NAME, token="cant_be_empty")

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
get_leaderboard()

## 9. Optional: Regenerate Existing Answers Too

In [ ]:
# Uncomment to regenerate every row, including rows that already had answers.
#
# results_all_regenerated = answer_qa_dataframe(qa_df, preserve_existing=False)
# results_all_regenerated.to_csv(WORK_DIR / "qa_set_api_rag_answers_regenerated.csv", sep=";", index=False)
# display(results_all_regenerated.head())

## 10. Inspect Low-Retrieval Questions

In [21]:
# This helps tune SIMILARITY_THRESHOLD and MAX_CONTEXT_CHUNKS.
# If many top scores are below 0.15, lower SIMILARITY_THRESHOLD to 0.12.
# If answers need more context, raise MAX_CONTEXT_CHUNKS to 8.

diagnostics = []
for question in qa_df["question"]:
    retrieved = retrieve_chunks(question)
    diagnostics.append(
        {
            "question": question,
            "top_score": retrieved[0]["score"] if retrieved else 0.0,
            "num_chunks_selected": len(retrieved),
            "top_source": retrieved[0]["source"] if retrieved else "",
            "top_chunk": retrieved[0]["chunk_index"] if retrieved else "",
        }
    )

diag_df = pd.DataFrame(diagnostics).sort_values("top_score")
display(diag_df.head(10))

,question,top_score,num_chunks_selected,top_source,top_chunk
18,Of the questions highlighted from the Member S...,0.114576,3,Web Version _E-Government Survey 2024 11102024.md,97
15,When were the simplified four stages of E-gove...,0.116146,3,Web Version _E-Government Survey 2024 11102024.md,43
16,Between what years was the EGDI revision 3.0 a...,0.119716,3,Web Version _E-Government Survey 2024 11102024.md,196
12,What is the lower bound of dead or missing whi...,0.127681,3,natural-catastrophe-and-climate-report-2023.md,0
31,What proportion of countries offer judiciary s...,0.127787,3,Web Version _E-Government Survey 2024 11102024.md,185
39,How many of the top 10 costliest environmental...,0.129976,3,natural-catastrophe-and-climate-report-2023.md,124
14,"According to survey data, which feature of gov...",0.130564,3,Web Version _E-Government Survey 2024 11102024.md,186
34,How has Japan named their initiative for remov...,0.135521,3,Web Version _E-Government Survey 2024 11102024.md,294
21,What grouping of nations has the closest OSI s...,0.141454,3,Web Version _E-Government Survey 2024 11102024.md,11
29,What fully digital service for individuals in ...,0.147097,3,Web Version _E-Government Survey 2024 11102024.md,18
